In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CKPT = "reformer-chatbot-checkpoints"  # folder from training
tok = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForCausalLM.from_pretrained(CKPT)
model.eval()

SYSTEM_PREFIX = ""  # keep empty; Reformer was trained on user/assistant tags
history = []

def format_dialog(history, user_msg):
    text = "".join(history) + f"<|user|> {user_msg}\n<|assistant|> "
    return text

@torch.inference_mode()
def chat(user_msg, max_new_tokens=120, temperature=0.8, top_p=0.9):
    prompt = format_dialog(history, user_msg)
    input_ids = tok(prompt, return_tensors="pt").input_ids
    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    full = tok.decode(out[0], skip_special_tokens=False)
    reply = full[len(prompt):]
    # trim at next user tag if exists
    stop = reply.find("<|user|>")
    reply = reply if stop == -1 else reply[:stop]
    history.append(f"<|user|> {user_msg}\n<|assistant|> {reply}")
    return reply.strip()

if __name__ == "__main__":
    print("Reformer Chatbot. Type 'exit' to quit.")
    while True:
        msg = input("You: ").strip()
        if msg.lower() == "exit": break
        print("Bot:", chat(msg))
